<a href="https://colab.research.google.com/github/Sbolivar16/MolecularDocking/blob/main/5_PharmacophoreToolkit_PostDocking_Colab_EN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pharmacophore-Toolkit — 3D pharmacophore from a docking pose
### Google Colab: protein PDB + docking pose + ligand SDF/MOL/MOL2

This notebook uses **pharmacophore-toolkit** (tlint101), based on RDKit, to generate a simple pharmacophore model from a docking pose.

## Inputs

1. **Protein:** `.pdb` file
2. **Docking pose:** `.pdb`, `.pdbqt`, `.sdf`, `.mol2`, or `.mol`
3. **Ligand reference structure:** preferably `.sdf`

> The third file is used to recover the correct connectivity, aromaticity, and bond orders.  
> The 3D coordinates are preserved from the docking pose.

## Outputs

- Table containing pharmacophoric features
- 3D coordinates for each feature
- Interactive 3D visualization
- 2D pharmacophore image
- `.pml` file for PyMOL
- Prepared protein–ligand complex
- Optional PLIP analysis
- ZIP file containing all results

## Workflow

**Protein + docking pose + ligand chemical information**  
→ pose correction  
→ **Pharmacophore-Toolkit**  
→ Donor / Acceptor / Aromatic / Hydrophobe  
→ 3D visualization  
→ export  
→ optional PLIP analysis to contextualize interactions with the receptor

### Main reference
https://github.com/tlint101/pharmacophore-toolkit


## 0. Installation


In [ ]:
#@title 0. Install dependencies
import sys, subprocess, shutil

def run(cmd, check=True):
    print(f"\n> {cmd}")
    p = subprocess.run(cmd, shell=True, text=True,
                       stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT)
    print(p.stdout)
    if check and p.returncode != 0:
        raise RuntimeError(f"Falló:\n{cmd}")
    return p

print("Python:", sys.version)

run("apt-get -qq update")
run("apt-get -qq install -y openbabel")

run(f"{sys.executable} -m pip install -U pip setuptools wheel")
run(f"{sys.executable} -m pip install -U rdkit pandas numpy py3Dmol pillow matplotlib")
run(f"{sys.executable} -m pip install -U pharmacophore-toolkit")

# PLIP is optional. If it fails, the pharmacophore workflow will still work.
plip = run(f"{sys.executable} -m pip install -U plip", check=False)
if plip.returncode != 0:
    print("⚠️ PLIP no se pudo instalar. Se continuará sin PLIP.")

print("\n✅ Installation completed.")


## 1. Verification


In [ ]:
#@title 1. Verify installation
import shutil, importlib

mods = ["rdkit", "pandas", "numpy", "py3Dmol", "pharmacophore"]
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"{m:20s}: ✅ OK")
    except Exception as e:
        print(f"{m:20s}: ❌ {e}")

print(f"{'obabel':20s}: {shutil.which('obabel')}")
print(f"{'plip':20s}: {shutil.which('plip')}")


## 2. Create working directories


In [ ]:
#@title 2. Directorios
from pathlib import Path
import shutil, subprocess, os, numpy as np, pandas as pd

ROOT = Path("/content/PharmacophoreToolkit_PostDocking")
INPUT = ROOT / "INPUT"
PREP = ROOT / "PREP"
TABLES = ROOT / "TABLES"
FIGS = ROOT / "FIGS"
PLIP_DIR = ROOT / "PLIP"
RESULTS = ROOT / "RESULTS"

for d in [ROOT, INPUT, PREP, TABLES, FIGS, PLIP_DIR, RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Working directories created:", ROOT)


## 3. Upload the protein PDB


In [ ]:
#@title 3. Upload protein
from google.colab import files

print("Select the protein in PDB format:")
uploaded = files.upload()

pdbs = [n for n in uploaded if n.lower().endswith(".pdb")]
if not pdbs:
    raise ValueError("No PDB file was uploaded.")

PROTEIN_PDB = INPUT / "protein.pdb"
shutil.move(pdbs[0], PROTEIN_PDB)

print("✅ Protein:", PROTEIN_PDB)


## 4. Upload the docking pose

Accepted formats: `.pdb`, `.pdbqt`, `.sdf`, `.mol2`, or `.mol`.

The geometry of this structure will be preserved as the 3D docking pose.


In [ ]:
#@title 4. Upload docking pose
print("Select the docking pose:")
uploaded = files.upload()

allowed = {".pdb", ".pdbqt", ".sdf", ".mol2", ".mol"}
poses = [n for n in uploaded if Path(n).suffix.lower() in allowed]

if not poses:
    raise ValueError("No compatible docking pose was uploaded.")

DOCKING_POSE = INPUT / f"docking_pose{Path(poses[0]).suffix.lower()}"
shutil.move(poses[0], DOCKING_POSE)

print("✅ Pose:", DOCKING_POSE)


## 5. Upload the ligand reference structure

Recommended: an **SDF** file with correct chemical information.

This file is used to recover correct bond orders and aromaticity, while the coordinates are taken from the docking pose.


In [ ]:
#@title 5. Upload ligand reference
print("Select an SDF, MOL, or MOL2 file:")
uploaded = files.upload()

allowed = {".sdf", ".mol", ".mol2"}
refs = [n for n in uploaded if Path(n).suffix.lower() in allowed]

if not refs:
    raise ValueError("An SDF/MOL/MOL2 file is required.")

LIGAND_REF = INPUT / f"ligand_reference{Path(refs[0]).suffix.lower()}"
shutil.move(refs[0], LIGAND_REF)

print("✅ Reference:", LIGAND_REF)


## 6. Read the ligand reference structure

RDKit reads the connectivity and generates a reference SMILES.


In [ ]:
#@title 6. Read ligand reference
from rdkit import Chem

ext = LIGAND_REF.suffix.lower()

if ext == ".sdf":
    supplier = Chem.SDMolSupplier(str(LIGAND_REF), removeHs=False, sanitize=True)
    ref_mol = next((m for m in supplier if m is not None), None)
elif ext == ".mol":
    ref_mol = Chem.MolFromMolFile(str(LIGAND_REF), removeHs=False, sanitize=True)
elif ext == ".mol2":
    ref_mol = Chem.MolFromMol2File(str(LIGAND_REF), removeHs=False, sanitize=True)
else:
    ref_mol = None

if ref_mol is None:
    raise ValueError("RDKit could not read the ligand reference structure.")

REFERENCE_SMILES = Chem.MolToSmiles(
    Chem.RemoveHs(ref_mol),
    canonical=True,
    isomericSmiles=True
)

print("✅ SMILES:", REFERENCE_SMILES)
print("Heavy atoms:", ref_mol.GetNumHeavyAtoms())


## 7. Convert the pose to SDF/PDB while preserving geometry

The pose is converted with Open Babel without using `--gen3d`; therefore, a new conformation is not generated.


In [ ]:
#@title 7. Normalize docking pose
POSE_SDF = PREP / "docking_pose_raw.sdf"
POSE_PDB = PREP / "docking_pose_raw.pdb"

# SDF
if DOCKING_POSE.suffix.lower() == ".sdf":
    shutil.copy(DOCKING_POSE, POSE_SDF)
else:
    p = subprocess.run(
        f'obabel "{DOCKING_POSE}" -O "{POSE_SDF}"',
        shell=True, text=True, capture_output=True
    )
    print(p.stdout, p.stderr)
    if p.returncode != 0:
        raise RuntimeError("The docking pose could not be converted to SDF.")

# PDB
if DOCKING_POSE.suffix.lower() == ".pdb":
    shutil.copy(DOCKING_POSE, POSE_PDB)
else:
    p = subprocess.run(
        f'obabel "{DOCKING_POSE}" -O "{POSE_PDB}"',
        shell=True, text=True, capture_output=True
    )
    print(p.stdout, p.stderr)
    if p.returncode != 0:
        raise RuntimeError("The docking pose could not be converted to PDB.")

print("✅ SDF:", POSE_SDF)
print("✅ PDB:", POSE_PDB)


## 8. Correct the docking-pose bond orders using the reference ligand

The goal is to preserve the docking coordinates while using the correct chemistry from the reference SDF.


In [ ]:
#@title 8. Reconstruct chemically correct pose
from rdkit import Chem
from rdkit.Chem import AllChem

pose_supplier = Chem.SDMolSupplier(
    str(POSE_SDF),
    removeHs=False,
    sanitize=False
)
pose_raw = next((m for m in pose_supplier if m is not None), None)

if pose_raw is None:
    raise ValueError("RDKit no pudo leer la pose.")

template = Chem.RemoveHs(ref_mol)
pose_noH = Chem.RemoveHs(pose_raw, sanitize=False)

try:
    pose_fixed = AllChem.AssignBondOrdersFromTemplate(template, pose_noH)
    Chem.SanitizeMol(pose_fixed)
    print("✅ Bond orders transferred successfully.")
except Exception as e:
    print("⚠️ Bond orders could not be transferred automatically.")
    print("Details:", e)
    print("The pose will be used as interpreted by RDKit.")
    pose_fixed = pose_noH
    try:
        Chem.SanitizeMol(pose_fixed)
    except:
        pass

POSE_FIXED_SDF = PREP / "docking_pose_fixed.sdf"
writer = Chem.SDWriter(str(POSE_FIXED_SDF))
writer.write(pose_fixed)
writer.close()

print("✅ Corrected pose:", POSE_FIXED_SDF)
print("Conformers:", pose_fixed.GetNumConformers())


## 9. Generate the pharmacophore with pharmacophore-toolkit

By default, the toolkit uses the following feature types:

- **Donor**
- **Acceptor**
- **Aromatic**
- **Hydrophobe**


In [ ]:
#@title 9. Calculate pharmacophore
from pharmacophore import Pharmacophore, Draw, View

pharm = Pharmacophore()

print("Available features:")
print(pharm.feature_types())

# The documentation recommends working without explicit H atoms for clarity
query_mol = Chem.RemoveHs(pose_fixed)

pharma_points = pharm.calc_pharm(query_mol)

print(f"\n✅ Number of features: {len(pharma_points)}")
for i, feat in enumerate(pharma_points, 1):
    print(i, feat)


## 10. Convert the pharmacophore to a table

Each record contains:

- feature type
- atom indices
- X, Y, Z coordinates in Å


In [ ]:
#@title 10. Pharmacophore table
rows = []

for i, feat in enumerate(pharma_points, 1):
    # documented format:
    # [feature_type, atom_indices, x, y, z]
    rows.append({
        "point_id": f"F{i}",
        "feature_type": feat[0],
        "atom_indices": ",".join(map(str, feat[1])),
        "x_A": float(feat[2]),
        "y_A": float(feat[3]),
        "z_A": float(feat[4]),
    })

ph4_df = pd.DataFrame(rows)
PH4_CSV = TABLES / "pharmacophore_points.csv"
ph4_df.to_csv(PH4_CSV, index=False)

display(ph4_df)

print("\n✅ Saved:", PH4_CSV)


## 11. Summary of pharmacophoric features


In [ ]:
#@title 11. Summary
summary_df = (
    ph4_df.groupby("feature_type")
    .size()
    .reset_index(name="n_points")
    .sort_values("n_points", ascending=False)
)

SUMMARY_CSV = TABLES / "pharmacophore_summary.csv"
summary_df.to_csv(SUMMARY_CSV, index=False)

display(summary_df)


## 12. Interactive 3D visualization

The library provides `View()` to visualize the ligand and its pharmacophoric features using py3Dmol.


In [ ]:
#@title 12. Visualization with Pharmacophore-Toolkit
viewer = View(type="jupyter")

viewer.view(
    [query_mol],
    [pharma_points],
    labels=True,
    window=(800, 600)
)


## 13. Generate a PML file for PyMOL

The toolkit can export the pharmacophoric spheres as a `.pml` script.


In [ ]:
#@title 13. Export PML
PML_OUT = RESULTS / "pharmacophore_features.pml"

# calc_pharm stores the features in the Pharmacophore object
pharm.output_features(savepath=str(PML_OUT))

print("✅ PML:", PML_OUT)


## 14. Generate a 2D representation

Atoms associated with pharmacophoric features are highlighted on the ligand.


In [ ]:
#@title 14. 2D pharmacophore
draw = Draw()
img = draw.draw_pharm(query_mol)

PNG_2D = FIGS / "pharmacophore_2D.png"

try:
    img.save(str(PNG_2D))
    print("✅ Image saved:", PNG_2D)
except Exception:
    print("The function displayed the figure directly; it could not be saved automatically.")

display(img)


## 15. Build the protein–ligand complex

A combined PDB file is generated for visual inspection and optional PLIP analysis.


In [ ]:
#@title 15. PDB complex
LIGAND_PDB = PREP / "ligand_fixed.pdb"

Chem.MolToPDBFile(query_mol, str(LIGAND_PDB))

protein_lines = PROTEIN_PDB.read_text().splitlines()
ligand_lines = LIGAND_PDB.read_text().splitlines()

clean_protein = [l for l in protein_lines if not l.startswith(("END", "CONECT"))]

# Rename the ligand as LIG, chain Z, residue 9999
clean_lig = []
serial_start = 1

# find maximum atom serial
for l in clean_protein:
    if l.startswith(("ATOM  ", "HETATM")):
        try:
            serial_start = max(serial_start, int(l[6:11]) + 1)
        except:
            pass

serial = serial_start
for l in ligand_lines:
    if not l.startswith(("ATOM  ", "HETATM")):
        continue
    l = l.ljust(80)
    atom_name = l[12:16]
    altloc = l[16:17]
    xyzrest = l[30:]
    new_line = f'HETATM{serial:5d} {atom_name}{altloc}{"LIG":>3s} Z{9999:4d}    {xyzrest}'
    clean_lig.append(new_line[:80])
    serial += 1

COMPLEX_PDB = PREP / "protein_ligand_complex.pdb"
COMPLEX_PDB.write_text("\n".join(clean_protein + clean_lig + ["END"]) + "\n")

print("✅ Complex:", COMPLEX_PDB)


## 16. Visualize protein + ligand + pharmacophore


In [ ]:
#@title 16. Integrated visualization
import py3Dmol

colors = {
    "Donor": "blue",
    "Acceptor": "red",
    "Aromatic": "gold",
    "Hydrophobe": "green"
}

view = py3Dmol.view(width=1000, height=650)
view.addModel(COMPLEX_PDB.read_text(), "pdb")
view.setStyle({"protein": True}, {"cartoon": {"color": "lightgray"}})
view.setStyle({"resn": "LIG", "chain": "Z"}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.25}})

for _, r in ph4_df.iterrows():
    center = {"x": float(r.x_A), "y": float(r.y_A), "z": float(r.z_A)}
    color = colors.get(r.feature_type, "magenta")
    view.addSphere({
        "center": center,
        "radius": 0.75,
        "color": color,
        "opacity": 0.65
    })
    view.addLabel(
        f'{r.point_id}:{r.feature_type}',
        {
            "position": center,
            "fontSize": 10,
            "backgroundColor": color,
            "fontColor": "black"
        }
    )

view.zoomTo({"resn": "LIG", "chain": "Z"})
view.show()

print("Legend:")
for k,v in colors.items():
    print(f"{k:12s}: {v}")


## 17. Optional PLIP analysis: protein–ligand interactions

If PLIP was successfully installed, this cell identifies non-covalent interactions in the complex.

**Note:** the pharmacophore above is generated by pharmacophore-toolkit. PLIP is used here as complementary information to help interpret which pharmacophoric features are located in an interaction environment with the receptor.


In [ ]:
#@title 17. Run PLIP
import shutil, subprocess

plip_exe = shutil.which("plip")

if plip_exe is None:
    print("⚠️ PLIP is not installed. Skip this cell.")
else:
    cmd = f'plip -f "{COMPLEX_PDB}" -x -t -o "{PLIP_DIR}"'
    print(">", cmd)

    result = subprocess.run(
        cmd, shell=True, text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )

    print(result.stdout[-6000:])

    if result.returncode == 0:
        print("✅ PLIP completed.")
    else:
        print("⚠️ PLIP finished with an error. The pharmacophore was generated successfully.")


## 18. Interpretation

The generated model represents 3D chemical features of the ligand in its docking geometry:

- **Donor:** potential hydrogen-bond donor
- **Acceptor:** potential hydrogen-bond acceptor
- **Aromatic:** center of an aromatic system
- **Hydrophobe:** hydrophobic region

### Important

The toolkit identifies the **pharmacophoric features of the ligand**, but by itself it does not prove that every feature is interacting with the receptor.

For a more complete interaction analysis, it is useful to combine it with PLIP/ProLIF or information obtained from molecular dynamics simulations.

### Recommended workflow

**Docking**  
→ **Pharmacophore-Toolkit**  
→ 3D pharmacophoric features  
→ **PLIP / ProLIF**  
→ protein–ligand interpretation  
→ **Uni-GBSA**  
→ **Molecular Dynamics**  
→ MM/PBSA or MM/GBSA


## 19. Download results


In [ ]:
#@title 19. Download ZIP
from google.colab import files
import shutil

ZIP_BASE = "/content/PharmacophoreToolkit_PostDocking_Results"
zip_path = shutil.make_archive(ZIP_BASE, "zip", ROOT)

print("✅ ZIP:", zip_path)
files.download(zip_path)
